In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext jupyter_black

## Processing all inferences to get summary tables

In [ ]:
import logging
from pathlib import Path
import numpy as np
import polars as pl
from tqdm.notebook import tqdm

from longbel.error_analysis import load_predictions

def compute_paired_deltas(
    merged_df: pl.DataFrame,
    unique_labels: list[str],
    n_bootstrap: int = 1000,
    ci: float = 0.95,
    seed: int = 42
) -> dict[str, dict[str, float]]:
    rng = np.random.default_rng(seed)
    doc_ids = merged_df["doc_id"].to_numpy() if merged_df.height else np.array([], dtype=object)
    unique_docs = np.unique(doc_ids)

    results = {}
    if merged_df.height == 0:
        return results

    # --- Point estimates ---
    agg = merged_df.group_by("semantic_group").agg([
        pl.len().alias("count"),
        (pl.col("gold_concept_code") == pl.col("pred_concept_code")).sum().cast(pl.Int64).alias("ctx_correct"),
        (pl.col("gold_concept_code") == pl.col("pred_concept_code_base")).sum().cast(pl.Int64).alias("base_correct"),
    ]).with_columns([
        ((pl.col("ctx_correct") - pl.col("base_correct")) / pl.col("count")).alias("delta")
    ])

    for row in agg.iter_rows(named=True):
        lbl = row["semantic_group"]
        results[lbl] = {"delta_strict": row["delta"] * 100}

    ctx_tot = int((merged_df["gold_concept_code"] == merged_df["pred_concept_code"]).sum())
    base_tot = int((merged_df["gold_concept_code"] == merged_df["pred_concept_code_base"]).sum())
    results["overall"] = {"delta_strict": (ctx_tot - base_tot) / merged_df.height * 100}

    # --- Paired Bootstrap ---
    if len(unique_docs) > 0 and n_bootstrap > 0:
        doc_agg = merged_df.group_by(["doc_id", "semantic_group"]).agg([
            pl.len().alias("denom"),
            (pl.col("gold_concept_code") == pl.col("pred_concept_code")).sum().alias("ctx_num"),
            (pl.col("gold_concept_code") == pl.col("pred_concept_code_base")).sum().alias("base_num"),
        ])
        
        n_docs = len(unique_docs)
        weights = rng.multinomial(n_docs, [1.0/n_docs]*n_docs, size=n_bootstrap)
        docs_df = pl.DataFrame({"doc_id": unique_docs})

        def _safe_boot_quantiles(df_filtered):
            if df_filtered.height == 0:
                return float('nan'), float('nan')
            
            j = docs_df.join(df_filtered, on="doc_id", how="left").fill_null(0)
            d_vec = j["denom"].to_numpy()
            c_vec = j["ctx_num"].to_numpy()
            b_vec = j["base_num"].to_numpy()

            boot_denom = weights @ d_vec
            boot_ctx = weights @ c_vec
            boot_base = weights @ b_vec

            boot_delta = np.full_like(boot_denom, np.nan, dtype=np.float64)
            valid_mask = boot_denom > 0
            boot_delta[valid_mask] = (boot_ctx[valid_mask] - boot_base[valid_mask]) / boot_denom[valid_mask]

            alpha = (1 - ci) / 2
            return float(np.nanquantile(boot_delta, alpha)) * 100, float(np.nanquantile(boot_delta, 1 - alpha)) * 100

        doc_overall = doc_agg.group_by("doc_id").agg([
            pl.col("denom").sum().alias("denom"),
            pl.col("ctx_num").sum().alias("ctx_num"),
            pl.col("base_num").sum().alias("base_num"),
        ])
        
        ci_low, ci_high = _safe_boot_quantiles(doc_overall)
        results["overall"]["delta_strict_ci_low"] = ci_low
        results["overall"]["delta_strict_ci_high"] = ci_high

        for label in unique_labels:
            if label not in results:
                continue
            l_df = doc_agg.filter(pl.col("semantic_group") == label)
            ci_low, ci_high = _safe_boot_quantiles(l_df)
            results[label]["delta_strict_ci_low"] = ci_low
            results[label]["delta_strict_ci_high"] = ci_high

    return results

# Helper from earlier to generate the fusion in memory
def fuse_multiple_predictions(dfs: list[pl.DataFrame], k: int = 60, top_n: int = 5):
    if len(dfs) < 2:
        raise ValueError("Need at least 2 dataframes to fuse.")

    metadata_cols = [col for col in dfs[0].columns if col not in ["pred_concept_code", "score", "rank"]]
    mention_metadata = dfs[0].select(metadata_cols).unique(subset=["doc_id", "mention_id"])

    processed_dfs = []
    
    for i, df in enumerate(dfs):
        df_sorted = df.sort(["doc_id", "mention_id", "score"], descending=[False, False, True])
        df_ranked = df_sorted.with_columns(
            pl.col("score").rank("ordinal", descending=True).over(["doc_id", "mention_id"]).alias("rank")
        )
        df_scored = df_ranked.with_columns((1.0 / (k + pl.col("rank"))).alias(f"rrf_score_{i}"))
        df_scored = df_scored.rename({"score": f"score_{i}"})
        cols_to_keep = ["doc_id", "mention_id", "pred_concept_code", f"rrf_score_{i}", f"score_{i}"]
        processed_dfs.append(df_scored.select(cols_to_keep))

    df_fused = processed_dfs[0]
    for i in range(1, len(processed_dfs)):
        df_fused = df_fused.join(
            processed_dfs[i], on=["doc_id", "mention_id", "pred_concept_code"],
            how="full", coalesce=True
        )

    rrf_cols = [f"rrf_score_{i}" for i in range(len(dfs))]
    score_cols = [f"score_{i}" for i in range(len(dfs))]

    fill_exprs = []
    for rrf_col in rrf_cols:
        fill_exprs.append(pl.col(rrf_col).fill_null(0.0))
    for score_col in score_cols:
        fill_exprs.append(pl.col(score_col).fill_null(-float("inf")))
        
    df_fused = df_fused.with_columns(fill_exprs)
    df_fused = df_fused.with_columns(sum(pl.col(c) for c in rrf_cols).alias("fused_score"))

    sort_cols = ["doc_id", "mention_id", "fused_score"] + score_cols
    descending_flags = [False, False, True] + [True] * len(score_cols)
    
    df_fused = df_fused.sort(sort_cols, descending=descending_flags)
    df_fused = df_fused.with_columns(
        pl.col("fused_score").rank("ordinal", descending=True).over(["doc_id", "mention_id"]).alias("rank")
    )
    df_fused = df_fused.filter(pl.col("rank") <= top_n)
    df_fused = df_fused.rename({"fused_score": "score"})
    df_fused = df_fused.join(mention_metadata, on=["doc_id", "mention_id"], how="left")

    return df_fused


def main(datasets: list[str]):
    all_deltas = {}
    selection_method = "tfidf"
    model_names = ["Llama-3.2-1B-Instruct", "Llama-3.1-8B-Instruct"]
    baseline_context = "short"
    
    # NEW: We define "runs" to be computed. Single models are strings. 
    # Fused models are Tuples containing a sub-tuple of formats and an RRF k-value.
    runs_to_evaluate = [
        "hybrid_long", 
        "hybrid_long_v2", 
        "hybrid_short", 
        "hybrid_short_v2", 
        "hybrid_medium", 
        "long",
        
        # FUSED ENSEMBLES (just map what you used in evaluate_fused_models)
        (("hybrid_long_v2", "hybrid_short_v2", "hybrid_medium", "short"), 60), 
    ]

    for dataset in tqdm(datasets, desc="Datasets"):
        data_split = "test"
        
        train_path = Path("data") / "final_data" / dataset / f"train_{selection_method}_annotations_{baseline_context}.tsv"
        if not train_path.exists():
            continue
            
        train_df = pl.read_csv(train_path, separator="\t", has_header=True, schema_overrides={"gold_concept_code": str, "mention_id": str, "doc_id": str})
        val_path = Path("data") / "final_data" / dataset / f"validation_{selection_method}_annotations_{baseline_context}.tsv"
        if val_path.exists():
            val_df = pl.read_csv(val_path, separator="\t", has_header=True, schema_overrides={"gold_concept_code": str, "mention_id": str, "doc_id": str})
            val_df = val_df[:int(len(val_df) * 0.9)]
            train_df = pl.concat([train_df, val_df])
            
        train_cuis = set(train_df["gold_concept_code"].drop_nulls())
        train_mentions = set(train_df["mention"].drop_nulls())
        unique_pairs = set(train_df.select(["mention", "gold_concept_code"]).drop_nulls().unique().iter_rows())
        top_100_cuis = set(train_df["gold_concept_code"].value_counts().head(100)["gold_concept_code"])
        top_100_mentions = set(train_df["mention"].value_counts().head(100)["mention"])

        for model_name in tqdm(model_names, desc="Models", leave=False):
            for model_check in ["last", "best"]:
                for complete_str in ["", "_complete"]:
                    for add_headers_str in ["_addheaders"]:
                        for aug_data in ["human_only"]:
                            for constraint in [True]:
                                
                                # 1. LOAD BASELINE
                                base_path = (Path("results") / "inference_outputs" / dataset / 
                                    f"{aug_data}_{selection_method}_{baseline_context}{complete_str}{add_headers_str}" / 
                                    f"{model_name}_{model_check}" / f"pred_{data_split}_{'constraint' if constraint else 'no_constraint'}_5_beams.tsv")
                                
                                if not base_path.exists():
                                    continue
                                base_pred = load_predictions(base_path, top_k=1)
                                
                                # 2. RUN EVALUATION OVER MIXED TARGETS (Single & Fused)
                                for run_config in runs_to_evaluate:
                                    is_fused = isinstance(run_config, tuple)
                                    ctx_pred = None
                                    model_name_str = ""
                                    
                                    if is_fused:
                                        # Process an ensemble fusion
                                        formats_tuple, rrf_k = run_config
                                        dfs_to_fuse = []
                                        skip_combo = False
                                        
                                        for fmt in formats_tuple:
                                            fmt_path = (Path("results") / "inference_outputs" / dataset / 
                                                f"{aug_data}_{selection_method}_{fmt}{complete_str}{add_headers_str}" / 
                                                f"{model_name}_{model_check}" / f"pred_{data_split}_{'constraint' if constraint else 'no_constraint'}_5_beams.tsv")
                                            
                                            if not fmt_path.exists():
                                                skip_combo = True
                                                break
                                            
                                            # Keep top_k=5 for accurate RRF calculation!
                                            dfs_to_fuse.append(load_predictions(fmt_path, top_k=5))
                                            
                                        if skip_combo or len(dfs_to_fuse) < 2:
                                            continue
                                            
                                        # Perform fusion and extract top 1 result for delta calculation
                                        ctx_pred_raw = fuse_multiple_predictions(dfs_to_fuse, k=rrf_k)
                                        ctx_pred = ctx_pred_raw.filter(pl.col("rank") == 1)
                                        
                                        # Create the combined column name (e.g. short+hybrid_medium+hybrid_long_k60)
                                        combo_name = "+".join(formats_tuple)
                                        model_name_str = f"{model_name}_{aug_data}_{combo_name}_k{rrf_k}{complete_str}{add_headers_str}_constraint_{model_check}"
                                        
                                    else:
                                        # Process a standard single model
                                        context_format = run_config
                                        ctx_path = (Path("results") / "inference_outputs" / dataset / 
                                            f"{aug_data}_{selection_method}_{context_format}{complete_str}{add_headers_str}" / 
                                            f"{model_name}_{model_check}" / f"pred_{data_split}_{'constraint' if constraint else 'no_constraint'}_5_beams.tsv")
                                        
                                        if not ctx_path.exists():
                                            continue
                                        ctx_pred = load_predictions(ctx_path, top_k=1)
                                        model_name_str = f"{model_name}_{aug_data}_{context_format}{complete_str}{add_headers_str}_constraint_{model_check}"
                                    
                                    if ctx_pred is None:
                                        continue

                                    # 3. MERGE FOR PAIRED EVALUATION
                                    merged = ctx_pred.join(
                                        base_pred.select(["doc_id", "mention_id", "pred_concept_code"]),
                                        on=["doc_id", "mention_id"],
                                        suffix="_base"
                                    )
                                    
                                    if merged.height == 0:
                                        continue
                                        
                                    unique_labels = sorted(merged["semantic_group"].unique().to_list())
                                    
                                    merged = (
                                        merged.with_columns(pl.col("mention_id").str.split(".").list.get(1).cast(pl.Float64).alias("mention_order"))
                                        .sort(["doc_id", "mention_order"])
                                        .with_columns(pl.col("mention_id").cum_count().over(["doc_id", "gold_concept_code"]).alias("repeat_rank"))
                                    )
                                    repeated_df = merged.filter(pl.col("repeat_rank") >= 2)
                                    unique_df = pl.DataFrame(list(unique_pairs), schema=["mention", "gold_concept_code"], orient="row")

                                    partitions = [
                                        ("All", merged),
                                        ("seen_cuis", merged.filter(pl.col("gold_concept_code").is_in(list(train_cuis)))),
                                        ("unseen_cuis", merged.filter(~pl.col("gold_concept_code").is_in(list(train_cuis)))),
                                        ("seen_mentions", merged.filter(pl.col("mention").is_in(list(train_mentions)))),
                                        ("unseen_mentions", merged.filter(~pl.col("mention").is_in(list(train_mentions)))),
                                        ("in_top_100_cuis", merged.filter(pl.col("gold_concept_code").is_in(list(top_100_cuis)))),
                                        ("not_in_top_100_cuis", merged.filter(~pl.col("gold_concept_code").is_in(list(top_100_cuis)))),
                                        ("in_top_100_mentions", merged.filter(pl.col("mention").is_in(list(top_100_mentions)))),
                                        ("not_in_top_100_mentions", merged.filter(~pl.col("mention").is_in(list(top_100_mentions)))),
                                        ("seen_unique_pairs", merged.join(unique_df, on=["mention", "gold_concept_code"], how="inner")),
                                        ("unseen_unique_pairs", merged.join(unique_df, on=["mention", "gold_concept_code"], how="anti")),
                                        ("identical", merged.filter(pl.col("mention") == pl.col("gold_concept_name"))),
                                        ("not_identical", merged.filter(pl.col("mention") != pl.col("gold_concept_name"))),
                                        ("one_word", merged.filter(pl.col("mention").str.count_matches(" ") == 0)),
                                        ("two_words", merged.filter(pl.col("mention").str.count_matches(" ") == 1)),
                                        ("three_words", merged.filter(pl.col("mention").str.count_matches(" ") == 2)),
                                        ("multi_words", merged.filter(pl.col("mention").str.count_matches(" ") >= 1)),
                                        ("more_than_three_words", merged.filter(pl.col("mention").str.count_matches(" ") >= 3)),
                                        ("repeated", repeated_df),
                                        ("repeated_rank_2", repeated_df.filter(pl.col("repeat_rank") == 2)),
                                        ("repeated_rank_3", repeated_df.filter(pl.col("repeat_rank") == 3)),
                                        ("repeated_rank_4", repeated_df.filter(pl.col("repeat_rank") == 4)),
                                        ("repeated_rank_gte_5", repeated_df.filter(pl.col("repeat_rank") >= 5)),
                                        ("not_repeated", merged.filter(pl.col("repeat_rank") == 1)),
                                    ]
                                    
                                    for part_name, part_df in partitions:
                                        res = compute_paired_deltas(part_df, unique_labels, n_bootstrap=1000)
                                        
                                        for lbl, dict_res in res.items():
                                            if lbl not in all_deltas:
                                                all_deltas[lbl] = {}
                                            if dataset not in all_deltas[lbl]:
                                                all_deltas[lbl][dataset] = {}
                                                
                                            if part_name not in all_deltas[lbl][dataset]:
                                                all_deltas[lbl][dataset][part_name] = {"index": part_name}
                                                
                                            dict_target = all_deltas[lbl][dataset][part_name]
                                            dict_target[f"{model_name_str}_delta_strict"] = dict_res.get("delta_strict", np.nan)
                                            dict_target[f"{model_name_str}_delta_strict_ci_low"] = dict_res.get("delta_strict_ci_low", np.nan)
                                            dict_target[f"{model_name_str}_delta_strict_ci_high"] = dict_res.get("delta_strict_ci_high", np.nan)

    # Final Write
    for label in all_deltas.keys():
        for dataset in datasets:
            if dataset not in all_deltas[label]:
                continue
                
            rows = list(all_deltas[label][dataset].values())
            df_delta = pl.DataFrame(rows)
            
            delta_path = Path("results") / "error_analysis_models" / dataset / f"delta_{label}.csv"
            delta_path.parent.mkdir(parents=True, exist_ok=True)
            df_delta.write_csv(delta_path)
            
    print("✅ Paired delta computation complete! Files saved to results/error_analysis_models/<dataset>/delta_<label>.csv")

In [ ]:
datasets_to_evaluate = ["EMEA", "MedMentions","SPACCC"]
main(datasets=datasets_to_evaluate)

In [ ]:
import logging
from pathlib import Path

import polars as pl
from tqdm import tqdm

from longbel.error_analysis import compute_metrics, load_predictions

def main(datasets: list[str]):
    all_scores = {}
    all_ratios = {}
    selection_method = "tfidf"
    model_names = [
        "Llama-3.2-1B-Instruct",
        # "Llama-3.2-3B-Instruct",
        "Llama-3.1-8B-Instruct",
    ]
    for dataset in tqdm(datasets, desc="Evaluation"):
        data_split = "test"
        for context_format in ["short", "hybrid_long_v2"]:
            # Load train data for seen/unseen evaluation
            train_path = (
                Path("data")
                / "final_data"
                / dataset
                / f"train_{selection_method}_annotations_{context_format}.tsv"
            )
            if train_path.exists():
                train_df = pl.read_csv(
                    train_path,
                    separator="\t",
                    has_header=True,
                    schema_overrides={
                        "gold_concept_code": str,
                        "mention_id": str,
                        "doc_id": str,  # force as string
                    },  # type: ignore
                )
            else:
                continue
            if context_format not in ["hybrid_short_v2", "hybrid_long_v2"]:
                validation_path = (
                    Path("data")
                    / "final_data"
                    / dataset
                    / f"validation_{selection_method}_annotations_{context_format}.tsv"
                )
                if validation_path.exists():
                    val_df = pl.read_csv(
                        validation_path,
                        separator="\t",
                        has_header=True,
                        schema_overrides={
                            "gold_concept_code": str,
                            "mention_id": str,
                            "doc_id": str,  # force as string
                        },  # type: ignore
                    )
                    # Reduce validation dataset to 10% as before
                    split = int(len(val_df) * 0.9)
                    val_df = val_df[:split]
                    train_df = pl.concat([train_df, val_df])
            train_cuis = set(train_df["gold_concept_code"].drop_nulls())
            train_mentions = set(train_df["mention"].drop_nulls())
            unique_pairs = (
                train_df.select(["mention", "gold_concept_code"])
                .drop_nulls()
                .unique()
                .iter_rows()
            )
            unique_pairs = set(unique_pairs)
            top_100_cuis = set(
                train_df["gold_concept_code"]
                .value_counts()
                .head(100)["gold_concept_code"]
            )
            top_100_mentions = set(
                train_df["mention"].value_counts().head(100)["mention"]
            )
            preditction_path = None
            for add_headers_str in ["_addheaders"]:
                for complete_str in ["", "_complete"]:
                    for model_check in ["last", "best"]:
                        for model_name in model_names:
                            for aug_data in ["human_only"]:
                                for constraint in [True]:
                                    preditction_path = (
                                        Path("results")
                                        / "inference_outputs"
                                        / dataset
                                        / f"{aug_data}_{selection_method}_{context_format}{complete_str}{add_headers_str}"
                                        / f"{model_name}_{model_check}"
                                        / f"pred_{data_split}_{'no_constraint' if not constraint else 'constraint'}_5_beams.tsv"
                                    )
                                    model_name_str = f"{model_name}_{aug_data}_{context_format}{complete_str}{add_headers_str}_{'no_constraint' if not constraint else 'constraint'}_{model_check}"
                                    if not preditction_path.exists():
                                        logging.warning(
                                            f"Prediction file not found, skipping: {preditction_path}"
                                        )
                                        continue
                                    pred_df = load_predictions(
                                        preditction_path,
                                        top_k=5
                                    )
                                    compute_all_recalls = "LLM_Evaluation" in pred_df.columns
                                    scores = compute_metrics(
                                        pred_df=pred_df,
                                        train_mentions=train_mentions,
                                        train_cuis=train_cuis,
                                        top_100_cuis=top_100_cuis,
                                        top_100_mentions=top_100_mentions,
                                        unique_pairs=unique_pairs,
                                        compute_all_recalls=compute_all_recalls,
                                        bootstrap=True,
                                        n_bootstrap=1000,
                                    )
                                    for label in scores.keys():
                                        if label not in all_ratios:
                                            all_ratios[label] = {
                                                "index": scores[label]["index"]
                                            }
                                        if label not in all_scores:
                                            all_scores[label] = {}
                                        if dataset not in all_scores[label]:
                                            all_scores[label][dataset] = {
                                                "index": scores[label]["index"]
                                            }
                                        for recall_score in scores[label].keys():
                                            if recall_score.startswith("recall_"):
                                                model_name_str_label = f"{model_name_str}_{recall_score.replace('recall_', '')}"
                                                # if compute_all_recalls:
                                                #     model_name_str_label = f"{model_name_str}_{recall_score.replace('recall_', '')}"
                                                # else:
                                                #     model_name_str_label = model_name_str
                                                all_scores[label][dataset][
                                                    model_name_str_label
                                                ] = scores[label][recall_score]
                                        all_ratios[label][dataset] = scores[label]["ratios"]

    # Write results
    for label in all_scores.keys():
        for dataset in datasets:
            if dataset not in all_scores[label]:
                continue
            df_score = pl.DataFrame(all_scores[label][dataset])
            score_path = (
                Path("results")
                / "error_analysis_models"
                / dataset
                / f"score_{label}.csv"
            )
            score_path.parent.mkdir(parents=True, exist_ok=True)
            df_score.write_csv(score_path)
            if label == "overall":
                # Filter to get only the "All" row
                all_row = df_score.filter(pl.col("index") == "All")
                if not all_row.is_empty():
                    print(f"\nDataset: {dataset}")
                    print("-" * 50)
                    # Get all columns except 'index' and sort them alphabetically
                    sorted_columns = sorted([col for col in all_row.columns if col != "index"])
                    # Print each metric on a new line
                    for col in sorted_columns:
                        if col != "index":
                            value = all_row[col][0]
                            print(f"  {col}: {value:.4f}")
        df_ratio = pl.DataFrame(all_ratios[label])
        ratio_path = (
            Path("results") / "error_analysis_models" / "Ratios" / f"ratio_{label}.csv"
        )
        ratio_path.parent.mkdir(parents=True, exist_ok=True)
        df_ratio.write_csv(ratio_path)

In [ ]:
datasets_to_evaluate = ["SPACCC", "EMEA", "MedMentions"]
main(datasets=datasets_to_evaluate)

In [ ]:
import logging
from pathlib import Path
import polars as pl
from tqdm import tqdm
from longbel.error_analysis import compute_metrics, load_predictions

def fuse_multiple_predictions(dfs: list[pl.DataFrame], k: int = 60, top_n: int = 5):
    """
    Fuses a variable number of top-K prediction dataframes using Reciprocal Rank Fusion (RRF).
    `dfs` is a list of DataFrames to fuse.
    """
    if len(dfs) < 2:
        raise ValueError("Need at least 2 dataframes to fuse.")

    # Extract static metadata from the first dataframe 
    metadata_cols = [col for col in dfs[0].columns if col not in ["pred_concept_code", "score", "rank"]]
    mention_metadata = dfs[0].select(metadata_cols).unique(subset=["doc_id", "mention_id"])

    processed_dfs = []
    
    for i, df in enumerate(dfs):
        # 1. Rank calculation
        df_sorted = df.sort(["doc_id", "mention_id", "score"], descending=[False, False, True])
        df_ranked = df_sorted.with_columns(
            pl.col("score").rank("ordinal", descending=True).over(["doc_id", "mention_id"]).alias("rank")
        )
        
        # 2. RRF score calculation
        df_scored = df_ranked.with_columns((1.0 / (k + pl.col("rank"))).alias(f"rrf_score_{i}"))
        df_scored = df_scored.rename({"score": f"score_{i}"})
        
        cols_to_keep = ["doc_id", "mention_id", "pred_concept_code", f"rrf_score_{i}", f"score_{i}"]
        processed_dfs.append(df_scored.select(cols_to_keep))

    # 3. Concatenate all predictions iteratively using Outer Coalesce Join
    df_fused = processed_dfs[0]
    for i in range(1, len(processed_dfs)):
        df_fused = df_fused.join(
            processed_dfs[i],
            on=["doc_id", "mention_id", "pred_concept_code"],
            how="full",
            coalesce=True,
        )

    # Gather the dynamic column names for summing
    rrf_cols = [f"rrf_score_{i}" for i in range(len(dfs))]
    score_cols = [f"score_{i}" for i in range(len(dfs))]

    # Fill nulls
    fill_exprs = []
    for rrf_col in rrf_cols:
        fill_exprs.append(pl.col(rrf_col).fill_null(0.0))
    for score_col in score_cols:
        fill_exprs.append(pl.col(score_col).fill_null(-float("inf")))
        
    df_fused = df_fused.with_columns(fill_exprs)

    # 4. Calculate final fused score (sum of all RRF scores)
    df_fused = df_fused.with_columns(
        sum(pl.col(c) for c in rrf_cols).alias("fused_score")
    )

    # 5. Sort by new fused score and raw scores as tie-breakers
    sort_cols = ["doc_id", "mention_id", "fused_score"] + score_cols
    descending_flags = [False, False, True] + [True] * len(score_cols)
    
    df_fused = df_fused.sort(sort_cols, descending=descending_flags)
    
    # NEW: Create the exact official 1-to-N "rank" for the fused output
    df_fused = df_fused.with_columns(
        pl.col("fused_score").rank("ordinal", descending=True).over(["doc_id", "mention_id"]).alias("rank")
    )

    # Filter to top_n using the newly created rank column
    df_fused = df_fused.filter(pl.col("rank") <= top_n)
    
    # Rename fused_score back to score to maintain compatibility
    df_fused = df_fused.rename({"fused_score": "score"})

    # NEW: Re-attach the static metadata (gold codes, mention text, semantic_group)
    df_fused = df_fused.join(mention_metadata, on=["doc_id", "mention_id"], how="left")

    return df_fused

def evaluate_fused_models(datasets: list[str]):
    fused_scores = {}
    fused_ratios = {}
    selection_method = "tfidf"
    model_names = [
        "Llama-3.2-1B-Instruct",
        "Llama-3.1-8B-Instruct",
    ]
    
    # Define combinations as tuples: ( Tuple_of_formats, RRF_k )
    # This allows pairs, trios, or more!
    fusion_configs = [
        # PAIRS
        (("hybrid_medium",   "hybrid_long_v2"), 60),
        (("short",           "hybrid_long"),    60),
        (("short",           "hybrid_long_v2"), 60),
        (("short", "hybrid_medium", "hybrid_short_v2", "hybrid_long_v2"), 60),
        (("hybrid_long_v2", "hybrid_short_v2", "hybrid_medium", "short"), 60),
    ]
    
    for dataset in tqdm(datasets, desc="Evaluation (Fused)"):
        data_split = "test"
        
        # ... (Keep your train_df / val_df data loading exactly as before) ...
        train_path = (Path("data") / "final_data" / dataset / f"train_{selection_method}_annotations_short.tsv")
        if not train_path.exists():
            continue
            
        train_df = pl.read_csv(
            train_path, separator="\t", has_header=True,
            schema_overrides={"gold_concept_code": str, "mention_id": str, "doc_id": str}
        )
        
        validation_path = (Path("data") / "final_data" / dataset / f"validation_{selection_method}_annotations_short.tsv")
        if validation_path.exists():
            val_df = pl.read_csv(
                validation_path, separator="\t", has_header=True,
                schema_overrides={"gold_concept_code": str, "mention_id": str, "doc_id": str}
            )
            split = int(len(val_df) * 0.9)
            train_df = pl.concat([train_df, val_df[:split]])
            
        train_cuis = set(train_df["gold_concept_code"].drop_nulls())
        train_mentions = set(train_df["mention"].drop_nulls())
        unique_pairs = set(train_df.select(["mention", "gold_concept_code"]).drop_nulls().unique().iter_rows())
        top_100_cuis = set(train_df["gold_concept_code"].value_counts().head(100)["gold_concept_code"])
        top_100_mentions = set(train_df["mention"].value_counts().head(100)["mention"])

        for add_headers_str in ["_addheaders"]:
            for complete_str in ["", "_complete"]:
                for model_check in ["last", "best"]:
                    for model_name in model_names:
                        for aug_data in ["human_only"]:
                            for constraint in [True]:
                                
                                # Loop over specific fusion configurations (tuples of formats, and k)
                                for formats_tuple, rrf_k in fusion_configs:
                                    
                                    dfs_to_fuse = []
                                    skip_combo = False
                                    
                                    # Load all dataframes for this combination
                                    for fmt in formats_tuple:
                                        fmt_path = (
                                            Path("results") / "inference_outputs" / dataset 
                                            / f"{aug_data}_{selection_method}_{fmt}{complete_str}{add_headers_str}" 
                                            / f"{model_name}_{model_check}" 
                                            / f"pred_{data_split}_{'no_constraint' if not constraint else 'constraint'}_5_beams.tsv"
                                        )
                                        
                                        if not fmt_path.exists():
                                            skip_combo = True
                                            break
                                            
                                        dfs_to_fuse.append(load_predictions(fmt_path, top_k=5))
                                        
                                    if skip_combo or len(dfs_to_fuse) < 2:
                                        continue
                                        
                                    # Fuse dynamically!
                                    df_fused = fuse_multiple_predictions(dfs_to_fuse, k=rrf_k)
                                    
                                    # Create dynamic name: e.g. "short+hybrid_medium+hybrid_long_k60"
                                    combo_name = "+".join(formats_tuple)
                                    fused_model_name_str = f"{model_name}_{aug_data}_{combo_name}_k{rrf_k}{complete_str}{add_headers_str}_{'no_constraint' if not constraint else 'constraint'}_{model_check}"
                                    
                                    compute_all_recalls = "LLM_Evaluation" in df_fused.columns
                                    
                                    scores = compute_metrics(
                                        pred_df=df_fused,
                                        train_mentions=train_mentions,
                                        train_cuis=train_cuis,
                                        top_100_cuis=top_100_cuis,
                                        top_100_mentions=top_100_mentions,
                                        unique_pairs=unique_pairs,
                                        compute_all_recalls=compute_all_recalls,
                                        bootstrap=True,
                                        n_bootstrap=1000,
                                    )
                                    
                                    for label in scores.keys():
                                        if label not in fused_ratios:
                                            fused_ratios[label] = {"index": scores[label]["index"]}
                                        if label not in fused_scores:
                                            fused_scores[label] = {}
                                        if dataset not in fused_scores[label]:
                                            fused_scores[label][dataset] = {"index": scores[label]["index"]}
                                            
                                        for recall_score in scores[label].keys():
                                            if recall_score.startswith("recall_"):
                                                label_key = f"{fused_model_name_str}_{recall_score.replace('recall_', '')}"
                                                fused_scores[label][dataset][label_key] = scores[label][recall_score]
                                                
                                        fused_ratios[label][dataset] = scores[label]["ratios"]

    # Write Fused Results
    for label in fused_scores.keys():
        for dataset in datasets:
            if dataset not in fused_scores[label]:
                continue
            df_score = pl.DataFrame(fused_scores[label][dataset])
            score_path = (Path("results") / "error_analysis_models" / dataset / f"score_fused_{label}.csv")
            score_path.parent.mkdir(parents=True, exist_ok=True)
            df_score.write_csv(score_path)
            
            if label == "overall":
                all_row = df_score.filter(pl.col("index") == "All")
                if not all_row.is_empty():
                    print(f"\n[FUSED] Dataset: {dataset}")
                    sorted_columns = sorted([col for col in all_row.columns if col != "index"])
                    for col in sorted_columns:
                        if col != "index":
                            print(f"  {col}: {all_row[col][0]:.4f}")


In [ ]:
datasets_to_evaluate = ["SPACCC", "EMEA", "MedMentions"]
evaluate_fused_models(datasets=datasets_to_evaluate)

## Semantic Group Plot

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Config
# -------------------------
ROOT = Path("results/error_analysis_models")
KEEP_AUG_DATA = "human_only"
FIXED_INDEX = "repeated"  # Your new delta partitions use lowercase 'repeated'

MODEL_OPTIONS = [
    ("Llama 1B", "Llama-3.2-1B-Instruct"),
    ("Llama 8B", "Llama-3.1-8B-Instruct"),
]
AUG_KNOWN = ["human_only_ft", "full_upsampled", "human_only"]

PREFERRED_CONTEXT_ORDER = [
    "hybrid_long_v2",
    "hybrid_long",
    "hybrid_short_v2",
    "hybrid_short",
    "hybrid_medium",
    "long",
]

# The exact dataset names we expect as columns in the Ratios table
DATASETS = ["SPACCC", "EMEA", "MedMentions", "MEDLINE"]

# -------------------------
# Helpers
# -------------------------
def norm_text(s):
    return str(s).strip().lower()

def translate_label(label: str) -> str:
    mapping = {
        "overall": "Overall",
        # Shortened SPACCC Labels
        "ENFERMEDAD": "Disease",
        "SINTOMA": "Symptom",
        "PROCEDIMIENTO": "Procedure",
        # Shortened MedMentions Labels
        "Organizations": "Orgs.",
        "Geographic Areas": "Geo. Areas",
        "Concepts & Ideas": "Concepts",
        "Genes & Molecular Sequences": "Genes & Mol.",
        "Living Beings": "Organisms",
        "Chemicals & Drugs": "Chem. & Drugs",
    }
    return mapping.get(label, label)

def parse_delta_run_id(run_id: str):
    m = re.search(r"_(no_constraint|constraint)_(best|last)$", run_id)
    if not m: return None
    
    constraint = m.group(1)
    checkpoint = m.group(2)
    prefix = run_id[: m.start()]
    
    if "_" not in prefix: return None
    model_name, rest = prefix.split("_", 1)
    
    aug_data = None
    for aug in AUG_KNOWN:
        if rest.startswith(aug + "_"):
            aug_data = aug
            ctx_flags = rest[len(aug) + 1 :]
            break
    if aug_data is None: return None

    context_format = ctx_flags
    for suffix in ["_complete_addheaders", "_addheaders", "_complete"]:
        if context_format.endswith(suffix):
            context_format = context_format[: -len(suffix)]
            break

    return {
        "model_name": model_name,
        "aug_data": aug_data,
        "context_format": context_format,
        "base_run": run_id[: -(len(checkpoint) + 1)],
    }

def load_n_supports(root: Path):
    """
    Scans the Ratios/ directory for 'Ratio_{label}.csv'.
    Locates the 'Repeated' row. 
    Matches each dataset (column) to extract numerator A from '% (A/B)'.
    """
    supports = {}
    ratios_dir = root / "Ratios"
    if not ratios_dir.exists():
        print(f"Warning: {ratios_dir} not found. Label n-counts will be missing.")
        return supports
        
    for p in ratios_dir.glob("*.csv"):
        if not p.name.lower().startswith("ratio_"):
            continue
            
        raw_label = p.stem[len("ratio_"):] if p.stem.lower().startswith("ratio_") else p.stem
        label_str = translate_label(raw_label)
        
        try:
            df = pl.read_csv(p)
            if df.is_empty(): continue
            
            # Find the index column generically (first column is usually 'index' or '')
            idx_col = df.columns[0]
            
            repeated_row = df.filter(pl.col(idx_col).cast(pl.Utf8).str.to_lowercase() == "repeated")
            if repeated_row.height > 0:
                row_dict = repeated_row.to_dicts()[0]
                
                # Check EVERY expected dataset column explicitely
                for dataset_nm in DATASETS:
                    # Ratios table columns might be lowercase or exact match, we do a case-insensitive check
                    matching_col = next((c for c in df.columns if c.lower() == dataset_nm.lower()), None)
                    
                    if matching_col and matching_col in row_dict:
                        val = row_dict[matching_col]
                        m = re.search(r"\((\d+)\s*/", str(val))
                        if m:
                            val_a = int(m.group(1))
                            # Map using the exact expected dataset name for lookup later
                            supports[(dataset_nm, label_str)] = val_a

        except Exception as e:
            print(f"Failed to parse support from {p.name}: {e}")
            
    return supports

def load_delta_rows(root: Path):
    rows = []
    for score_path in root.glob("*/delta_*.csv"):
        dataset = score_path.parent.name
        if dataset.upper() == "MEDLINE":
            continue  
            
        label = translate_label(score_path.stem.replace("delta_", "", 1))
        df = pl.read_csv(score_path)
        if "index" not in df.columns: continue
        
        cols = df.columns
        delta_cols = [c for c in cols if c.endswith("_delta_strict") and not c.endswith("_ci_low") and not c.endswith("_ci_high")]

        for row in df.iter_rows(named=True):
            idx_val = str(row["index"])
            for dc in delta_cols:
                run_id = dc[: -len("_delta_strict")]
                low_col = run_id + "_delta_strict_ci_low"
                high_col = run_id + "_delta_strict_ci_high"

                rows.append({
                    "dataset": dataset,
                    "label": label,
                    "index": idx_val,
                    "run_id": run_id,
                    "delta": float(row[dc]) if row[dc] is not None else np.nan,
                    "delta_ci_low": float(row[low_col]) if (low_col in cols and row[low_col] is not None) else np.nan,
                    "delta_ci_high": float(row[high_col]) if (high_col in cols and row[high_col] is not None) else np.nan,
                })
    return pd.DataFrame(rows)

def prepare_best_deltas(root: Path):
    raw = load_delta_rows(root)
    if raw.empty:
        raise ValueError("No delta CSV files found. Make sure running the paired script generated data.")
    
    parsed = raw["run_id"].apply(parse_delta_run_id)
    mask = parsed.notna()
    df = pd.concat([raw[mask].reset_index(drop=True), pd.DataFrame(parsed[mask].tolist()).reset_index(drop=True)], axis=1)

    df = df[df["aug_data"].eq(KEEP_AUG_DATA)]
    df = df[df["model_name"].isin([v for _, v in MODEL_OPTIONS])]
    df = df[df["delta"].notna()]

    idx_best = df.groupby(["dataset", "label", "index", "model_name", "context_format"])["delta"].idxmax()
    df_best = df.loc[idx_best].copy()
    
    df_best["is_overall"] = df_best["label"].map(lambda x: norm_text(x) == "overall")
    return df_best

def get_label_order_ascending(df):
    if df.empty: return []
    non_overall = df[~df["is_overall"]].sort_values(by="label", ascending=True)
    overall = df[df["is_overall"]]
    ordered = non_overall["label"].tolist() + overall["label"].tolist()
    seen = set()
    return [x for x in ordered if not (x in seen or seen.add(x))]

def format_label_with_n(label_str, dataset_name, supports):
    n_val = supports.get((dataset_name, label_str))
    if n_val is not None:
        if "Overall" in label_str:
            return f"{label_str} (N={n_val})"
        return f"{label_str} (n={n_val})"
    return label_str

def draw_ci_boxes(ax, df, label_order, crop_min, crop_max, title, dataset_name, supports):
    ax.axvline(0, color="#444", lw=1.2, zorder=0)
    ax.set_xlim(crop_min, crop_max)
    ax.grid(axis="x", color="#e0e0e0", linestyle="--", linewidth=0.8, zorder=0)

    ax.set_title(title, loc='center', fontsize=14, fontweight='bold', pad=12)

    y_map = {lab: i for i, lab in enumerate(label_order)}
    n = len(label_order)
    
    formatted_labels = [format_label_with_n(lbl, dataset_name, supports) for lbl in label_order]
    
    ax.set_yticks(np.arange(n))
    # Rotation removed as requested, keeping horizontally aligned
    ax.set_yticklabels(formatted_labels, fontsize=12)
    ax.set_ylim(n - 0.5, -0.5)

    for tick_label in ax.get_yticklabels():
        if "Overall" in tick_label.get_text():
            tick_label.set_fontweight('bold')
            
    if "Overall" in y_map:
        y_over = y_map["Overall"]
        ax.axhspan(y_over - 0.5, y_over + 0.5, facecolor="#f0f4f8", zorder=-1)
        
    margin = (crop_max - crop_min) * 0.015

    for _, r in df.iterrows():
        if r["label"] not in y_map: continue
        y = y_map[r["label"]]

        low, high, d = r["delta_ci_low"], r["delta_ci_high"], r["delta"]
        is_overall = r["is_overall"]

        l_clip, h_clip, d_clip = max(low, crop_min), min(high, crop_max), min(max(d, crop_min), crop_max)

        if np.isfinite(l_clip) and np.isfinite(h_clip) and h_clip >= l_clip:
            ax.plot([l_clip, h_clip], [y, y], color="#888888", lw=2.5, zorder=1)
            cap_h = 0.25
            ax.plot([l_clip, l_clip], [y - cap_h, y + cap_h], color="#888888", lw=2, zorder=1)
            ax.plot([h_clip, h_clip], [y - cap_h, y + cap_h], color="#888888", lw=2, zorder=1)

        marker, color, size = ("D", "#1f4e79", 110) if is_overall else ("o", "#1f77b4", 90)
        ax.scatter([d_clip], [y], s=size, marker=marker, color=color, edgecolor="black", zorder=3)

        if low < crop_min: 
            ax.plot(crop_min + margin, y, marker="<", color="black", markersize=9, zorder=4, linestyle="None")
        if high > crop_max: 
            ax.plot(crop_max - margin, y, marker=">", color="black", markersize=9, zorder=4, linestyle="None")

def build_figure(model, context, crop_min, crop_max, df_best, supports):
    target_data = df_best[
        (df_best["model_name"] == model) &
        (df_best["context_format"] == context) &
        (df_best["index"].map(norm_text) == norm_text(FIXED_INDEX))
    ].copy()

    # Reordered datasets extraction
    d_mm = target_data[target_data["dataset"] == "MedMentions"]
    d_emea = target_data[target_data["dataset"] == "EMEA"]
    d_sp = target_data[target_data["dataset"] == "SPACCC"]

    if d_mm.empty and d_emea.empty and d_sp.empty:
        return None

    ord_mm = get_label_order_ascending(d_mm)
    ord_emea = get_label_order_ascending(d_emea)
    ord_sp = get_label_order_ascending(d_sp)

    # Reordered height computation
    h_mm = max(2.0, 0.4 * len(ord_mm))
    h_emea = max(2.0, 0.4 * len(ord_emea))
    h_sp = max(2.0, 0.4 * len(ord_sp))

    fig = plt.figure(figsize=(6, h_mm + h_emea + h_sp + 1.5))
    gs = fig.add_gridspec(3, 1, height_ratios=[h_mm, h_emea, h_sp], hspace=0.25)

    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[2, 0])

    if not d_mm.empty: draw_ci_boxes(ax1, d_mm, ord_mm, crop_min, crop_max, "MedMentions-ST21pv", "MedMentions", supports)
    if not d_emea.empty: draw_ci_boxes(ax2, d_emea, ord_emea, crop_min, crop_max, "QUAERO-EMEA", "EMEA", supports)
    if not d_sp.empty: draw_ci_boxes(ax3, d_sp, ord_sp, crop_min, crop_max, "SPACCC", "SPACCC", supports)

    ax3.set_xlabel(r"$\Delta$ Recall@1 (LongBEL - Baseline)", fontsize=14, fontweight='bold', labelpad=12)
    for ax in [ax1, ax2, ax3]: ax.tick_params(axis='x', labelsize=11)

    fig.subplots_adjust(left=0.25, right=0.95, top=0.95, bottom=0.12)
    return fig

# -------------------------
# Build data once
# -------------------------
df_best = prepare_best_deltas(ROOT)
print("Parsing Sample Sizes (N counts) per dataset from Ratios/ ...")
support_counts = load_n_supports(ROOT)
clear_output()

# -------------------------
# Widgets & UI
# -------------------------
available_models = sorted(df_best["model_name"].unique())
model_opts = [(lbl, val) for (lbl, val) in MODEL_OPTIONS if val in available_models] or [(m, m) for m in available_models]
contexts = sorted(set(df_best["context_format"].dropna().astype(str).unique().tolist()))
pref_contexts = [c for c in PREFERRED_CONTEXT_ORDER if c in contexts] + [c for c in contexts if c not in PREFERRED_CONTEXT_ORDER]

model_dd = widgets.Dropdown(options=model_opts, value=model_opts[0][1], description="Model:")
context_dd = widgets.Dropdown(options=pref_contexts, value=pref_contexts[0], description="Context:")
crop_min_dd = widgets.FloatText(value=-20.0, description="Crop min:")
crop_max_dd = widgets.FloatText(value=20.0, description="Crop max:")

save_btn = widgets.Button(description="Save to PDF", icon="save", button_style="success")
save_out = widgets.Output()

def render_plot(model, context, crop_min, crop_max):
    fig = build_figure(model, context, crop_min, crop_max, df_best, support_counts)
    if fig: plt.show()
    else: print("No data to display. Check matching filters.")

def on_save_clicked(b):
    with save_out:
        save_out.clear_output()
        m, c = model_dd.value, context_dd.value
        cmin, cmax = crop_min_dd.value, crop_max_dd.value
        
        fig = build_figure(m, c, cmin, cmax, df_best, support_counts)
        if fig:
            filename = f"deltas_{m.replace(' ', '_')}_{c}.pdf"
            fig.savefig(filename, bbox_inches='tight', dpi=300)
            plt.close(fig) 
            print(f"✅ Successfully saved: {filename}")
        else:
            print("❌ Nothing to save (Empty data).")

save_btn.on_click(on_save_clicked)

ui_row = widgets.HBox([model_dd, context_dd, crop_min_dd, crop_max_dd, save_btn])
out_plot = widgets.interactive_output(render_plot, {"model": model_dd, "context": context_dd, "crop_min": crop_min_dd, "crop_max": crop_max_dd})

display(ui_row)
display(save_out)
display(out_plot)

## Delta Plot

In [ ]:
from pathlib import Path
import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# -------------------------
# Config
# -------------------------
DATASETS = ["SPACCC", "EMEA", "MedMentions", "MEDLINE"]
MODELS = ["Llama-3.2-1B-Instruct", "Llama-3.1-8B-Instruct"]

AUG_DATA = "human_only"
SELECTION_METHOD = "tfidf"
ADDHEADERS_STR = "_addheaders"
COMPLETE_STR = ""
CONSTRAINT_STR = "constraint"
SPLIT = "test"
NUM_BEAMS = 5

# Soft color palette
COLOR_IMP = "#82ca9d"  # Soft Green (Improvement / < 0)
COLOR_DEG = "#f08080"  # Soft Red (Degradation / > 0)
COLOR_NAN = "#cccccc"  # Soft Gray
OVERALL_BG = "#f0f4f8"

# -------------------------
# Polars Data Loaders & Helpers (Strict Determinism)
# -------------------------
def normalize_codes_expr(col_name: str) -> pl.Expr:
    return (
        pl.col(col_name).cast(pl.Utf8)
        .str.split("+").list.eval(pl.element().str.strip_chars())
        .list.sort().list.join("+").alias(col_name)
    )

def mention_order_expr() -> pl.Expr:
    return (
        pl.col("mention_id").cast(pl.Utf8)
        .str.extract(r"(\d+)$", 1).cast(pl.Int64, strict=False)
        .fill_null(10**9).alias("mention_order")
    )

def build_pred_path(dataset: str, model: str, context: str, checkpoint: str) -> Path:
    return (Path("results") / "inference_outputs" / dataset / 
            f"{AUG_DATA}_{SELECTION_METHOD}_{context}{COMPLETE_STR}{ADDHEADERS_STR}" / 
            f"{model}_{checkpoint}" / f"pred_{SPLIT}_{CONSTRAINT_STR}_{NUM_BEAMS}_beams.tsv")

def load_top1(path: Path) -> pl.DataFrame:
    df = pl.read_csv(path, separator="\t", has_header=True, schema_overrides={"doc_id": str, "mention_id": str, "gold_concept_code": str, "pred_concept_code": str, "semantic_group": str, "rank": pl.Int64})
    return (df.filter(pl.col("rank") == 1)
        .with_columns([normalize_codes_expr("gold_concept_code"), normalize_codes_expr("pred_concept_code"), mention_order_expr()])
        .unique(subset=["doc_id", "mention_id"], keep="first", maintain_order=True)
        .sort(["doc_id", "mention_order", "mention_id"]))

def strict_acc(df: pl.DataFrame) -> float:
    if df.height == 0: return float("nan")
    return float(df.filter(pl.col("pred_concept_code") == pl.col("gold_concept_code")).height / df.height)

def choose_best_checkpoint(dataset: str, model: str, context: str):
    candidates = []
    for ckpt in ["best", "last"]:
        p = build_pred_path(dataset, model, context, ckpt)
        if p.exists():
            d = load_top1(p)
            if d.height > 0: candidates.append((ckpt, p, d, strict_acc(d)))
    if not candidates: return None
    candidates.sort(key=lambda x: (x[3], x[0]), reverse=True)
    return candidates[0]

# --- 100% DETERMINISTIC COMPUTATION (METRIC A) ---
def compute_metric_a_strict(df: pl.DataFrame) -> pl.DataFrame:
    df = df.sort(["doc_id", "mention_order", "mention_id"]).with_row_index("row_nr")
    
    df = df.with_columns([
        ((pl.col("pred_concept_code") != pl.col("gold_concept_code")) & 
         pl.col("pred_concept_code").is_not_null() & 
         (pl.col("pred_concept_code") != "")).alias("is_err")
    ])

    first_errs = (
        df.filter(pl.col("is_err"))
          .sort("row_nr")
          .group_by(["doc_id", "gold_concept_code"], maintain_order=True)
          .first()
          .select([
              pl.col("doc_id"), 
              pl.col("gold_concept_code"), 
              pl.col("row_nr").alias("first_err_row"),
              pl.col("pred_concept_code").alias("first_err_pred_code")
          ])
    )

    df = df.join(first_errs, on=["doc_id", "gold_concept_code"], how="left")
    df = df.with_columns([(pl.col("row_nr") > pl.col("first_err_row")).fill_null(False).alias("exp_specific")])
    df = df.with_columns([(pl.col("exp_specific") & (pl.col("pred_concept_code") == pl.col("first_err_pred_code")) & pl.col("is_err")).alias("echo_err")])

    aggs = [
        pl.len().alias("n_total"),
        pl.col("exp_specific").sum().alias("n_exp_specific"),
        pl.col("echo_err").sum().alias("n_echo_err")
    ]
    
    label_res = df.group_by("semantic_group").agg(aggs).sort("semantic_group")
    overall_res = df.select(aggs).with_columns(pl.lit("Overall").alias("semantic_group")).select(label_res.columns)
    
    return pl.concat([label_res, overall_res], how="vertical").with_columns([
        (pl.col("n_echo_err") / pl.col("n_exp_specific") * 100).alias("echo_err_rate")
    ])

# -------------------------
# Build Aggregated Data
# -------------------------
cache_metric_a_strict = []

def get_strict_memory_data():
    if not cache_metric_a_strict:
        rows = []
        for dataset in ["SPACCC", "EMEA", "MedMentions"]: 
            for model in MODELS:
                for context in ["hybrid_long", "hybrid_long_v2"]:
                    chosen = choose_best_checkpoint(dataset, model, context)
                    if not chosen: continue
                    pl_res = compute_metric_a_strict(chosen[2])
                    df_res = pl_res.to_pandas()
                    df_res["dataset"] = dataset
                    df_res["model"] = model
                    df_res["context"] = context
                    rows.append(df_res)
        cache_metric_a_strict.extend(rows)
    return pd.concat(cache_metric_a_strict, ignore_index=True)

# -------------------------
# Visualization: Final Styled Bar Plot
# -------------------------
def translate_label(label: str) -> str:
    mapping = {
        "overall": "Overall",
        # Shortened SPACCC Labels
        "ENFERMEDAD": "Disease",
        "SINTOMA": "Symptom",
        "PROCEDIMIENTO": "Procedure",
        # Shortened MedMentions Labels
        "Organizations": "Orgs.",
        "Geographic Areas": "Geo. Areas",
        "Concepts & Ideas": "Concepts",
        "Genes & Molecular Sequences": "Genes & Mol.",
        "Living Beings": "Organisms",
        "Chemicals & Drugs": "Chem. & Drugs",
    }
    return mapping.get(label, label)

def draw_styled_bars(ax, df_sub, title, crop_min, crop_max):
    overall_mask = df_sub["semantic_group"] == "Overall"
    overall_df = df_sub[overall_mask]
    label_df = df_sub[~overall_mask].sort_values(by="semantic_group", ascending=True)
    d_ordered = pd.concat([label_df, overall_df]).reset_index(drop=True)
    
    y_pos = np.arange(len(d_ordered))
    deltas = d_ordered["delta"].values
    
    colors = [COLOR_IMP if pd.notna(v) and v < 0 else (COLOR_DEG if pd.notna(v) and v > 0 else COLOR_NAN) for v in deltas]
    capped_deltas = [min(crop_max, max(crop_min, v)) if pd.notna(v) else np.nan for v in deltas]
    
    ax.barh(y_pos, capped_deltas, color=colors, height=0.6, align='center', edgecolor='black', linewidth=0.5, zorder=2)
    ax.axvline(0, color='#444444', linewidth=1.2, zorder=1)
    
    for y, raw_val, rendered, color in zip(y_pos, deltas, capped_deltas, colors):
        if pd.isna(raw_val): continue
        
        if raw_val > 0:
            # Add > sign directly inside the text if it overflows
            val_str = f"+{raw_val:.1f}%"
            if raw_val > crop_max: val_str += " ▶"
            
            text_x = rendered - 0.2 if rendered >= 2.5 else rendered / 2
            ha_val = 'right' if rendered >= 2.5 else 'center'
            ax.text(text_x, y, val_str, va='center', ha=ha_val, color='white', fontweight='bold', fontsize=9, zorder=3)
                
        elif raw_val < 0:
            # Add < sign directly inside the text if it overflows
            val_str = f"{raw_val:.1f}%"
            if raw_val < crop_min: val_str = "◀ " + val_str
            
            text_x = rendered + 0.2 if rendered <= -2.5 else rendered / 2
            ha_val = 'left' if rendered <= -2.5 else 'center'
            ax.text(text_x, y, val_str, va='center', ha=ha_val, color='white', fontweight='bold', fontsize=9, zorder=3)
        else:
            ax.text(0, y, "0.0%", va='center', ha='center', color='black', fontweight='bold', fontsize=9, zorder=3)

    n = len(d_ordered)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(d_ordered["label"], fontsize=12)
    
    # By explicitly setting ylim here, we eliminate the padding above/below the plots
    # This closes the gap between the axhspan background and the border
    ax.set_ylim(n - 0.5, -0.5)
    
    ax.set_xlim(crop_min, crop_max)
    ax.grid(axis='x', color="#e0e0e0", linestyle="--", linewidth=0.8, zorder=0)
    ax.set_title(title, loc='center', fontsize=14, fontweight='bold', pad=12)
    
    if not overall_df.empty:
        overall_idx = n - 1
        ax.axhspan(overall_idx - 0.5, overall_idx + 0.5, facecolor=OVERALL_BG, zorder=0)
        ax.get_yticklabels()[overall_idx].set_fontweight("bold")
        
    ax.tick_params(axis='x', labelsize=11)

def build_figure(model, crop_min, crop_max, df_data):
    pivot_rate = df_data.pivot(index=["dataset", "semantic_group"], columns="context", values="echo_err_rate").reset_index()
    pivot_exp = df_data.pivot(index=["dataset", "semantic_group"], columns="context", values="n_exp_specific").reset_index()
    
    if "hybrid_long" not in pivot_rate.columns or "hybrid_long_v2" not in pivot_rate.columns:
        return None

    pivot = pivot_rate.merge(pivot_exp[["dataset", "semantic_group", "hybrid_long"]], on=["dataset", "semantic_group"], how="outer", suffixes=("", "_support"))
    pivot["delta"] = pivot["hybrid_long_v2"] - pivot["hybrid_long"]
    
    def format_label(row):
        base_label = translate_label(row["semantic_group"])
        n_val = int(row["hybrid_long_support"]) if pd.notnull(row["hybrid_long_support"]) else 0
        return f"{base_label} (N={n_val})" if row["semantic_group"] == "Overall" else f"{base_label} (n={n_val})"
        
    pivot["label"] = pivot.apply(format_label, axis=1)

    d_mm = pivot[pivot["dataset"] == "MedMentions"]
    d_emea = pivot[pivot["dataset"] == "EMEA"]
    d_sp = pivot[pivot["dataset"] == "SPACCC"]

    if d_mm.empty and d_emea.empty and d_sp.empty:
        return None

    h_mm = max(2.0, 0.4 * len(d_mm)) if not d_mm.empty else 0.001
    h_emea = max(2.0, 0.4 * len(d_emea)) if not d_emea.empty else 0.001
    h_sp = max(2.0, 0.4 * len(d_sp)) if not d_sp.empty else 0.001

    fig = plt.figure(figsize=(6, h_mm + h_emea + h_sp + 1.5))
    gs = fig.add_gridspec(3, 1, height_ratios=[h_mm, h_emea, h_sp], hspace=0.25)
    
    ax1 = fig.add_subplot(gs[0, 0])
    ax2 = fig.add_subplot(gs[1, 0])
    ax3 = fig.add_subplot(gs[2, 0])

    if not d_mm.empty: draw_styled_bars(ax1, d_mm, "MedMentions-ST21pv", crop_min, crop_max)
    if not d_emea.empty: draw_styled_bars(ax2, d_emea, "QUAERO-EMEA", crop_min, crop_max)
    if not d_sp.empty: draw_styled_bars(ax3, d_sp, "SPACCC", crop_min, crop_max)

    ax3.set_xlabel("Δ Copy Wrong Memory Error Rate", fontsize=14, fontweight='bold', labelpad=12)
    fig.subplots_adjust(left=0.25, right=0.95, top=0.95, bottom=0.12)
    
    return fig

# -------------------------
# UI & Main Execution
# -------------------------
print("Processing STRICT Metric A Data...")
df_mem = get_strict_memory_data()
clear_output()

model_dd = widgets.Dropdown(options=MODELS, value=MODELS[0], description="Model:")
crop_min_dd = widgets.FloatText(value=-10.0, description="Crop min:")
crop_max_dd = widgets.FloatText(value=10.0, description="Crop max:")

save_btn = widgets.Button(description="Save to PDF", icon="save", button_style="success")
save_out = widgets.Output()

def render_plot(model, crop_min, crop_max):
    df_subset = df_mem[df_mem["model"] == model]
    fig = build_figure(model, crop_min, crop_max, df_subset)
    if fig: plt.show()
    else: print("No valid data available to render.")

def on_save_clicked(b):
    with save_out:
        save_out.clear_output()
        m = model_dd.value
        cmin, cmax = float(crop_min_dd.value), float(crop_max_dd.value)
        df_subset = df_mem[df_mem["model"] == m]
        fig = build_figure(m, cmin, cmax, df_subset)
        if fig:
            filename = f"robustness_delta_{m.replace(' ', '_')}.pdf"
            fig.savefig(filename, bbox_inches='tight', dpi=300)
            plt.close(fig) 
            print(f"✅ Successfully saved: {filename}")
        else:
            print("❌ Nothing to save.")

save_btn.on_click(on_save_clicked)

ui_row = widgets.HBox([model_dd, crop_min_dd, crop_max_dd, save_btn])
out_plot = widgets.interactive_output(render_plot, {"model": model_dd, "crop_min": crop_min_dd, "crop_max": crop_max_dd})

display(ui_row)
display(save_out)
display(out_plot)